## Volatility Surface Visualisation

In [1]:
import pandas as pd
import plotly.graph_objects as go
import os
from dotenv import load_dotenv

# 1. LOAD CONFIGURATION & DATA
load_dotenv()
FILE_PATH = os.getenv("OPTIONS_DATA_FILE", "nifty_options_data.csv")

if not os.path.exists(FILE_PATH):
    print(f"❌ Error: File not found at {FILE_PATH}")
else:
    # Load Data
    df = pd.read_csv(FILE_PATH)
    
    # 2. DATA PRE-PROCESSING
    # Convert dates
    df['Datetime'] = pd.to_datetime(df['Datetime'])
    df['Expiry'] = pd.to_datetime(df['Expiry'])
    
    # Calculate Days to Expiry (DTE) for the Y-axis
    df['DTE'] = (df['Expiry'] - df['Datetime']).dt.days
    
    # Filter for cleaner visualization:
    # 1. IV between 1% and 150% (Remove outliers/bad data)
    # 2. DTE > 0 (Remove expired options)
    df_clean = df[(df['IV'] > 0.01) & (df['IV'] < 1.5) & (df['DTE'] > 0)].copy()

    # 3. SELECT A SPECIFIC DATE
    # A surface represents a snapshot in time. We must pick ONE single trade date.
    # We automatically pick the middle date of your dataset to ensure good data density.
    unique_dates = sorted(df_clean['Datetime'].dt.date.unique())
    selected_date = unique_dates[-1]
    
    # Create the subset for plotting
    surface_data = df_clean[df_clean['Datetime'].dt.date == selected_date].copy()
    
    print(f"✅ Loaded Data. Plotting Surface for Date: {selected_date}")
    print(f"📊 Data Points: {len(surface_data)} | Spot Price: {surface_data['SpotPrice'].iloc[0]}")

    # 4. CREATE INTERACTIVE PLOTLY FIGURE
    fig = go.Figure(data=[go.Mesh3d(
        x=surface_data['Strike'],
        y=surface_data['DTE'],
        z=surface_data['IV'] * 100,  # Convert to Percentage
        intensity=surface_data['IV'], # Color based on IV height
        colorscale='Viridis',
        opacity=0.8,
        flatshading=True
    )])

    # 5. LAYOUT SETTINGS
    fig.update_layout(
        title=dict(text=f"NIFTY Volatility Surface ({selected_date})", x=0.5),
        scene=dict(
            xaxis=dict(title='Strike Price (₹)'),
            yaxis=dict(title='Days to Expiry (DTE)'),
            zaxis=dict(title='Implied Volatility (%)'),
            aspectmode='manual', aspectratio=dict(x=1, y=1, z=0.7) # Adjust Z-height
        ),
        width=900,
        height=700,
        margin=dict(l=0, r=0, b=0, t=50) # Tight margins
    )

    # Show the interactive plot
    fig.show()

✅ Loaded Data. Plotting Surface for Date: 2026-04-24
📊 Data Points: 2056 | Spot Price: 23903.95


In [2]:
# ==========================================
# CELL 1: SETUP AND DATA LOADING
# Run this cell ONLY ONCE to load the dataset
# ==========================================
import sys, os
from dotenv import load_dotenv

# Add parent directory to path so we can import modules one level up
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from options_backtester import OptionsBacktester

load_dotenv()

# Point to the data directory in the global parent
DATA_FILE = os.getenv("OPTIONS_DATA_FILE", "../data/nifty_options_data.csv")

# Initialize the backtester engine
# Removed default_lot_size to use the 'LotSize' column from the CSV
backtester = OptionsBacktester(data_path=DATA_FILE)

print(f"✅ Data loaded from: {DATA_FILE}")
print(f"✅ Backtester initialized using dynamic LotSize from data.")

Initializing Backtester... Loading Data from data/NIFTY50-INDEX_opt_2017-07-17_2026-04-24.csv
Data successfully loaded and cleaned.
✅ Data loaded from: data/NIFTY50-INDEX_opt_2017-07-17_2026-04-24.csv
✅ Backtester initialized using dynamic LotSize from data.


In [3]:
# ==========================================
# CELL 2: STRATEGY DEFINITION & EXECUTION
# Tweak these variables and run as needed
# ==========================================

ANALYSIS_DATE = "2026-03-02"
TARGET_PNL_DATE = "2026-03-10"

# Sample Iron Condor Portfolio
portfolio = [
    {'strike': 25250, 'type': 'PE', 'action': 'buy',  'expiry': '2026-04-28', 'qty': 1}, 
    {'strike': 25300, 'type': 'PE', 'action': 'sell', 'expiry': '2026-04-28', 'qty': 1}, 
    {'strike': 25350, 'type': 'CE', 'action': 'sell', 'expiry': '2026-04-28', 'qty': 1}, 
    {'strike': 25400, 'type': 'CE', 'action': 'buy',  'expiry': '2026-04-28', 'qty': 1},
]

# Execute the Payoff Calculator
results = backtester.calculate_basket_payoff(
    portfolio=portfolio, 
    analysis_date=ANALYSIS_DATE, 
    target_pnl_date=TARGET_PNL_DATE,
    show_plot=True,
    verbose=True
)

# Safely extract the new status key
status = results.get("status")

# ---------------------------------------------------------
# OUTPUT FORMATTING & ERROR HANDLING
# ---------------------------------------------------------
if status == "error":
    print("\n" + "❌ " * 15)
    print("BACKTEST FAILED: DATA ERROR")
    print("-" * 35)
    for msg in results.get("messages", []):
        print(msg)
    print("❌ " * 15 + "\n")

elif status == "issue":
    print("\n" + "--- BACKTEST HALTED: TRADE ISSUE ---")
    for msg in results.get("messages", []):
        print(msg)
    print("------------------------------------\n")

elif status == "success":
    metrics = results['metrics']
    greeks = results['greeks']

    print("\n" + "="*40)
    print("STRATEGY SUMMARY")
    print("="*40)

    mp_disp = "Unlimited" if metrics['Max Profit'] == float('inf') else f"₹ {metrics['Max Profit']:,.2f}"
    ml_disp = "Unlimited" if metrics['Max Loss'] == float('-inf') else f"₹ {metrics['Max Loss']:,.2f}"

    print(f"{'Max Profit:':<18} {mp_disp}")
    print(f"{'Max Loss:':<18} {ml_disp}")
    print(f"{'Net Premium:':<18} ₹ {abs(metrics['Total Premium']):,.2f} {metrics['Credit/Debit']}")
    print(f"{'Breakevens:':<18} {[round(x, 2) for x in metrics['Breakevens']]}")
    print(f"{'Margin Required:':<18} ₹ {metrics['Margin Req']:,.2f}")
    print(f"{'ROI (%):':<18} {metrics['ROI (%)']:.2f}%")

    print("\n" + "-" * 40)
    print("NET GREEKS (Per Unit)")
    print("-" * 40)
    print(f"{'Delta:':<18} {greeks['delta']:.4f}")
    print(f"{'Gamma:':<18} {greeks['gamma']:.4f}")
    print(f"{'Theta:':<18} {greeks['theta']:.4f}")
    print(f"{'Vega:':<18} {greeks['vega']:.4f}")
    print("="*40)
else:
    print("\n⚠️ Unknown response from backtester:")
    print(results)


--- Strategy Breakdown | Analysis Day: 2026-03-02 | Spot: 24849.75 ---
BUY 25250.0 PE (2026-04-28) @ ₹577.10 | Qty: 1 | Lot: 65
SELL 25300.0 PE (2026-04-28) @ ₹605.75 | Qty: 1 | Lot: 65
SELL 25350.0 CE (2026-04-28) @ ₹437.35 | Qty: 1 | Lot: 65
BUY 25400.0 CE (2026-04-28) @ ₹408.60 | Qty: 1 | Lot: 65



STRATEGY SUMMARY
Max Profit:        ₹ 3,731.00
Max Loss:          ₹ 481.00
Net Premium:       ₹ 3,731.00 Credit
Breakevens:        []
Margin Required:   ₹ 65,283.88
ROI (%):           5.72%

----------------------------------------
NET GREEKS (Per Unit)
----------------------------------------
Delta:             -0.0001
Gamma:             -0.0000
Theta:             0.0027
Vega:              -0.1157


In [4]:
import pandas as pd
import numpy as np
import itertools
import plotly.graph_objects as go
from datetime import timedelta
from tqdm.auto import tqdm 

# ==========================================
# INPUTS
# ==========================================
BUYING_DATE_INPUT = '2026-03-06'
MAX_EXPIRY_DAYS_INPUT = 35
MAX_SHIFT = 400
MAX_SHAPE_SHIFT = 0 
MAX_RANGE_INPUT = 400 # <-- Added: Maximum points to widen the middle condor range

def calculate_basket_charges(legs):
    """
    Calculates realistic ENTRY charges for an options basket based on Fyers and Indian regulations.
    Matches actual contract note values. Assumes hold-to-expiry for exit (no exit brokerage).
    """
    buy_turnover = 0
    sell_turnover = 0
    
    for leg in legs:
        turnover = abs(leg['cash_premium'])
        if leg['action'].lower() == 'buy':
            buy_turnover += turnover
        else:
            sell_turnover += turnover
            
    total_turnover = buy_turnover + sell_turnover
    
    brokerage = 20 * len(legs)                   # 20 Rs per leg (80 Rs for 4 legs)
    stt = sell_turnover * 0.001                  # 0.1% STT on options sell side
    exc_txn = total_turnover * 0.0003503         # 0.03503% NSE transaction charge
    sebi = total_turnover * 0.000001             # ₹10 per crore
    stamp = buy_turnover * 0.00003               # 0.003% Stamp duty on buy side
    ipft = total_turnover * 0.000005             # ₹50 per crore
    
    gst = 0.18 * (brokerage + exc_txn + sebi)    # 18% GST on brokerage, txn, and SEBI
    
    total_entry_charges = brokerage + stt + exc_txn + sebi + stamp + ipft + gst
    return total_entry_charges

def find_arbitrage_baskets_plot(backtester, buying_date_str, max_expiry_days, max_range):
    buying_date = pd.to_datetime(buying_date_str).date()
    print(f"--- Initialization: Searching for Risk-Free Baskets on {buying_date} ---")

    df_source = backtester.df
    df_analysis = df_source[df_source['Date'] == pd.to_datetime(buying_date_str).normalize()].copy()
    
    if df_analysis.empty:
        print(f"❌ No data found for date: {buying_date}")
        return

    spot_price = df_analysis['SpotPrice'].iloc[0] if 'SpotPrice' in df_analysis.columns else df_analysis['Close'].iloc[0]
    atm_strike = round(spot_price / 50) * 50

    max_expiry_date = buying_date + timedelta(days=max_expiry_days)
    available_expiries = sorted([
        pd.to_datetime(d).date() for d in df_analysis['Expiry'].unique() 
        if buying_date < pd.to_datetime(d).date() <= max_expiry_date
    ])

    price_lookup = df_analysis.set_index([
        pd.to_datetime(df_analysis['Expiry']).dt.date, 
        df_analysis['Strike'].astype(float), 
        'Type'
    ])['Close'].to_dict()

    valid_options_set = set(price_lookup.keys())

    tasks = []
    base_shifts = range(-MAX_SHIFT, MAX_SHIFT + 50, 50)
    leg_variations = range(-MAX_SHAPE_SHIFT, MAX_SHAPE_SHIFT + 50, 50)
    shape_combinations = list(itertools.product(leg_variations, repeat=4))
    
    # <-- Added: Range extensions for the middle strikes
    range_extensions = range(0, max_range + 50, 50) 

    # Building tasks including the new range extension
    for expiry in available_expiries:
        for shift in base_shifts:
            for shape in shape_combinations:
                for r_ext in range_extensions:
                    tasks.append((expiry, shift, shape, r_ext))

    results = []
    tested_portfolios = set() 
    
    illiquid_count = 0
    charges_exceed_count = 0
    final_good_baskets = 0

    # Extracting r_ext from tasks
    for expiry, shift, shape, r_ext in tqdm(tasks, desc="Checking Baskets"):
        expiry_str = str(expiry)
        portfolio = []
        valid_legs = True
        current_strikes_list = []

        # <-- Added: Base legs dynamically generated per task iteration to include r_ext
        base_legs = [
            {'strike': atm_strike - 100,  'type': 'PE', 'action': 'buy'},
            {'strike': atm_strike - 50,   'type': 'PE', 'action': 'sell'},
            {'strike': atm_strike + 50 + r_ext,   'type': 'CE', 'action': 'sell'},
            {'strike': atm_strike + 100 + r_ext,  'type': 'CE', 'action': 'buy'}
        ]

        for i, base_leg in enumerate(base_legs):
            new_strike = float(base_leg['strike'] + shift + shape[i])
            current_strikes_list.append(int(new_strike))
            
            lookup_key = (expiry, new_strike, base_leg['type'])
            if lookup_key not in valid_options_set:
                valid_legs = False
                break
            
            portfolio.append({
                'strike': new_strike,
                'type': base_leg['type'],
                'action': base_leg['action'],
                'expiry': expiry_str,
                'qty': 1
            })
        
        if not valid_legs:
            continue
            
        unique_contracts = set((leg['strike'], leg['type']) for leg in portfolio)
        if len(unique_contracts) < 4:
            continue

        leg_sigs = tuple(sorted([(leg['strike'], leg['type'], leg['action']) for leg in portfolio]))
        portfolio_sig = (expiry_str, leg_sigs)
        if portfolio_sig in tested_portfolios:
            continue
        tested_portfolios.add(portfolio_sig)

        res = backtester.calculate_basket_payoff(
            portfolio=portfolio, 
            analysis_date=buying_date_str, 
            show_plot=False, 
            verbose=False
        )

        if res.get('status') == 'issue':
            illiquid_count += 1
            
        elif res.get('status') == 'success':
            metrics = res['metrics']
            
            if metrics['Max Loss'] > 0 and metrics['Max Profit'] > 0:
                # Check viability after transaction costs
                charges = calculate_basket_charges(res['legs'])
                min_profit = metrics['Max Loss']  # Minimum guaranteed payoff
                
                if charges >= min_profit:
                    charges_exceed_count += 1
                else:
                    final_good_baskets += 1
                    results.append({
                        'Expiry': expiry_str,
                        'Shift': shift,
                        'RangeExt': r_ext, # Track this if you need to analyze what ranges worked best later
                        'CenterStrike': sum(current_strikes_list) / 4,
                        'Strikes': str(current_strikes_list),
                        'Portfolio': res['legs'], 
                        'NetMinProfit': min_profit - charges,
                        'NetMaxProfit': metrics['Max Profit'] - charges,
                        'Charges': charges,
                        'ROI': metrics['ROI (%)'],
                        'Margin': metrics['Margin Req'],
                        'TotalPremium': metrics['Total Premium'], 
                        'CreditDebit': metrics['Credit/Debit']
                    })

    # Output Formatting Requirements
    print("\n" + "="*60)
    print("SEARCH RESULTS SUMMARY")
    print("="*60)
    print(f"Spot Price of Underlying: {spot_price:,.2f}")
    
    # Calculate total to ensure numbers add up linearly
    total_arbitrage_found = illiquid_count + charges_exceed_count + final_good_baskets
    
    print(f"Total Arbitrage Baskets Found: {total_arbitrage_found}")
    print(f"Baskets Discarded due to Illiquidity: {illiquid_count}")
    print(f"Baskets Discarded due to Charges > MinProfit: {charges_exceed_count}")
    print(f"Final Good Baskets Left: {final_good_baskets}")
    print("-" * 60)

    if not results:
        print("Search complete. No tradable risk-free baskets found after charges.")
        return

    # Create the dataframe and apply an explicit safety filter to ensure nothing sub-zero gets through to the plot
    results_df = pd.DataFrame(results)
    results_df = results_df[results_df['NetMinProfit'] > 0]
    
    if results_df.empty:
        print("Search complete. No tradable risk-free baskets found after charges.")
        return

    max_roi_row = results_df.loc[results_df['ROI'].idxmax()]

    def print_portfolio(row_data):
        for leg in row_data['Portfolio']:
            vol = 0
            if 'Volume' in df_analysis.columns:
                vol_mask = (df_analysis['Strike'] == leg['strike']) & \
                           (df_analysis['Type'] == leg['type']) & \
                           (pd.to_datetime(df_analysis['Expiry']).dt.date == pd.to_datetime(leg['expiry']).date())
                vol_data = df_analysis.loc[vol_mask, 'Volume']
                vol = int(vol_data.iloc[0]) if not vol_data.empty and not pd.isna(vol_data.iloc[0]) else 0
                
            print(f"      - {leg['action'].capitalize()} {leg['type']} @ {int(leg['strike'])} | Pts: ₹{leg['entry_price']:.2f} (Vol: {vol:,})")
        
        print(f"      => Net Premium: ₹{abs(row_data['TotalPremium']):,.2f} ({row_data['CreditDebit']})")

    print(f"🏆 BEST ROI BASKET:")
    print(f"   Expiry: {max_roi_row['Expiry']}")
    print_portfolio(max_roi_row)
    print(f"   ROI: {max_roi_row['ROI']:.2f}% | Margin: ₹{max_roi_row['Margin']:,.2f}")
    print(f"   Charges (Entry): ₹{max_roi_row['Charges']:,.2f}")
    print(f"   Net Guaranteed Profit: ₹{max_roi_row['NetMinProfit']:,.2f}")
    print("="*60 + "\n")

    print("Generating Plot...")
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=results_df['Expiry'],
        y=results_df['CenterStrike'],
        mode='markers',
        marker=dict(
            size=12,
            color=results_df['NetMinProfit'], 
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title="Net Guaranteed Profit (₹)")
        ),
        text=[
            f"<b>Expiry:</b> {r['Expiry']}<br>"
            f"<b>Avg Strike:</b> {r['CenterStrike']}<br>"
            f"<b>Strikes:</b> {r['Strikes']}<br>"
            f"<b>Mid Range Widen:</b> {r['RangeExt']}<br>" # <-- Added tracking to hover text
            f"<b>Charges:</b> ₹{r['Charges']:,.2f}<br>"
            f"<b>Net Min Profit:</b> ₹{r['NetMinProfit']:,.0f}<br>"
            f"<b>Net Max Profit:</b> ₹{r['NetMaxProfit']:,.0f}<br>"
            f"<b>Gross ROI:</b> {r['ROI']:.2f}%"
            for _, r in results_df.iterrows()
        ],
        hoverinfo='text',
        name='Risk-Free Baskets'
    ))

    fig.add_hline(y=spot_price, line_dash="dash", line_color="red", annotation_text=f"Spot: {spot_price}")

    fig.update_layout(
        title=f"Risk-Free Basket Opportunities (Buying Date: {buying_date} | Max {max_expiry_days} Days Out)",
        xaxis_title="Expiry Date",
        yaxis_title="Basket Avg Center Strike",
        height=700,
        template="plotly_white",
        xaxis=dict(type='category', categoryorder='category ascending')
    )

    fig.show()

# Execute the finder passing the new MAX_RANGE_INPUT
find_arbitrage_baskets_plot(backtester, BUYING_DATE_INPUT, MAX_EXPIRY_DAYS_INPUT, MAX_RANGE_INPUT)

--- Initialization: Searching for Risk-Free Baskets on 2026-03-06 ---


Checking Baskets:   0%|          | 0/765 [00:00<?, ?it/s]


SEARCH RESULTS SUMMARY
Spot Price of Underlying: 24,605.40
Total Arbitrage Baskets Found: 132
Baskets Discarded due to Illiquidity: 92
Baskets Discarded due to Charges > MinProfit: 10
Final Good Baskets Left: 30
------------------------------------------------------------
🏆 BEST ROI BASKET:
   Expiry: 2026-04-07
      - Buy PE @ 24350 | Pts: ₹353.55 (Vol: 3)
      - Sell PE @ 24400 | Pts: ₹434.85 (Vol: 43)
      - Sell CE @ 24650 | Pts: ₹528.25 (Vol: 11)
      - Buy CE @ 24700 | Pts: ₹456.50 (Vol: 11,242)
      => Net Premium: ₹9,948.25 (Credit)
   ROI: 14.19% | Margin: ₹70,122.45
   Charges (Entry): ₹206.93
   Net Guaranteed Profit: ₹6,491.32

Generating Plot...


In [5]:
import pandas as pd
import numpy as np
import itertools
import plotly.graph_objects as go
from datetime import timedelta
from tqdm.auto import tqdm 

# ==========================================
# INPUTS & HYPERPARAMETERS
# ==========================================
BUYING_DATE_INPUT = '2026-03-06'
MAX_EXPIRY_DAYS_INPUT = 15

# New parameters to control the combinatorial explosion
MAX_STRIKES_FROM_ATM = 8    # Checks this many strikes above and below ATM (e.g., 4 means 9 total strikes checked)
STRIKE_STEP = 50            # The gap between consecutive strikes in your underlying

def calculate_basket_charges(legs):
    """
    Calculates realistic ENTRY charges for an options basket based on Fyers and Indian regulations.
    Matches actual contract note values. Assumes hold-to-expiry for exit (no exit brokerage).
    """
    buy_turnover = 0
    sell_turnover = 0
    
    for leg in legs:
        turnover = abs(leg['cash_premium'])
        if leg['action'].lower() == 'buy':
            buy_turnover += turnover
        else:
            sell_turnover += turnover
            
    total_turnover = buy_turnover + sell_turnover
    
    brokerage = 20 * len(legs)                   # 20 Rs per leg (80 Rs for 4 legs)
    stt = sell_turnover * 0.001                  # 0.1% STT on options sell side
    exc_txn = total_turnover * 0.0003503         # 0.03503% NSE transaction charge
    sebi = total_turnover * 0.000001             # ₹10 per crore
    stamp = buy_turnover * 0.00003               # 0.003% Stamp duty on buy side
    ipft = total_turnover * 0.000005             # ₹50 per crore
    
    gst = 0.18 * (brokerage + exc_txn + sebi)    # 18% GST on brokerage, txn, and SEBI
    
    total_entry_charges = brokerage + stt + exc_txn + sebi + stamp + ipft + gst
    return total_entry_charges

def find_arbitrage_baskets_plot(backtester, buying_date_str, max_expiry_days):
    buying_date = pd.to_datetime(buying_date_str).date()
    print(f"--- Initialization: Searching ALL 4-Leg Combinations on {buying_date} ---")

    df_source = backtester.df
    df_analysis = df_source[df_source['Date'] == pd.to_datetime(buying_date_str).normalize()].copy()
    
    if df_analysis.empty:
        print(f"❌ No data found for date: {buying_date}")
        return

    spot_price = df_analysis['SpotPrice'].iloc[0] if 'SpotPrice' in df_analysis.columns else df_analysis['Close'].iloc[0]
    atm_strike = round(spot_price / STRIKE_STEP) * STRIKE_STEP

    max_expiry_date = buying_date + timedelta(days=max_expiry_days)
    available_expiries = sorted([
        pd.to_datetime(d).date() for d in df_analysis['Expiry'].unique() 
        if buying_date < pd.to_datetime(d).date() <= max_expiry_date
    ])

    price_lookup = df_analysis.set_index([
        pd.to_datetime(df_analysis['Expiry']).dt.date, 
        df_analysis['Strike'].astype(float), 
        'Type'
    ])['Close'].to_dict()

    valid_options_set = set(price_lookup.keys())
    tasks = []

    # Building the combinatorial tasks
    for expiry in available_expiries:
        expiry_str = str(expiry)
        
        # 1. Gather all valid strikes within our defined hyperparameter range for this expiry
        valid_strikes_in_range = [
            strike for (e, strike, opt_type) in valid_options_set 
            if e == expiry and abs(strike - atm_strike) <= (MAX_STRIKES_FROM_ATM * STRIKE_STEP)
        ]
        valid_strikes_in_range = sorted(list(set(valid_strikes_in_range)))
        
        # 2. Build a pool of all individual potential legs (Buy CE, Sell CE, Buy PE, Sell PE)
        leg_pool = []
        for strike in valid_strikes_in_range:
            for opt_type in ['CE', 'PE']:
                if (expiry, strike, opt_type) in valid_options_set:
                    leg_pool.append({'strike': strike, 'type': opt_type, 'action': 'buy'})
                    leg_pool.append({'strike': strike, 'type': opt_type, 'action': 'sell'})
                    
        # 3. Generate all combinations of 4 legs from the pool
        # This will test every possible structural combination 
        for combo in itertools.combinations(leg_pool, 4):
            # Enforce 4 DISTINCT contracts (e.g. don't buy and sell the exact same strike/type in one basket)
            unique_contracts = set((leg['strike'], leg['type']) for leg in combo)
            if len(unique_contracts) == 4:
                tasks.append((expiry_str, list(combo)))

    print(f"Total theoretical basket combinations to test: {len(tasks):,}")

    results = []
    tested_portfolios = set() 
    
    illiquid_count = 0
    charges_exceed_count = 0
    final_good_baskets = 0

    # Unpacking combinations from tasks
    for expiry_str, combo_legs in tqdm(tasks, desc="Checking Baskets"):
        portfolio = []
        
        for leg in combo_legs:
            portfolio.append({
                'strike': float(leg['strike']),
                'type': leg['type'],
                'action': leg['action'],
                'expiry': expiry_str,
                'qty': 1
            })

        leg_sigs = tuple(sorted([(leg['strike'], leg['type'], leg['action']) for leg in portfolio]))
        portfolio_sig = (expiry_str, leg_sigs)
        if portfolio_sig in tested_portfolios:
            continue
        tested_portfolios.add(portfolio_sig)

        res = backtester.calculate_basket_payoff(
            portfolio=portfolio, 
            analysis_date=buying_date_str, 
            show_plot=False, 
            verbose=False
        )

        if res.get('status') == 'issue':
            illiquid_count += 1
            
        elif res.get('status') == 'success':
            metrics = res['metrics']
            
            if metrics['Max Loss'] > 0 and metrics['Max Profit'] > 0:
                # Check viability after transaction costs
                charges = calculate_basket_charges(res['legs'])
                min_profit = metrics['Max Loss']  # Minimum guaranteed payoff
                
                if charges >= min_profit:
                    charges_exceed_count += 1
                else:
                    final_good_baskets += 1
                    
                    # Create a quick string summary of the legs for plotting
                    leg_details = ", ".join([f"{l['action'][0].upper()}{l['type']}{int(l['strike'])}" for l in portfolio])
                    avg_strike = sum([l['strike'] for l in portfolio]) / 4
                    
                    results.append({
                        'Expiry': expiry_str,
                        'CenterStrike': avg_strike,
                        'LegDetails': leg_details,
                        'Portfolio': res['legs'], 
                        'NetMinProfit': min_profit - charges,
                        'NetMaxProfit': metrics['Max Profit'] - charges,
                        'Charges': charges,
                        'ROI': metrics['ROI (%)'],
                        'Margin': metrics['Margin Req'],
                        'TotalPremium': metrics['Total Premium'], 
                        'CreditDebit': metrics['Credit/Debit']
                    })

    # Output Formatting Requirements
    print("\n" + "="*60)
    print("SEARCH RESULTS SUMMARY")
    print("="*60)
    print(f"Spot Price of Underlying: {spot_price:,.2f}")
    
    total_arbitrage_found = illiquid_count + charges_exceed_count + final_good_baskets
    
    print(f"Total Arbitrage Baskets Found: {total_arbitrage_found}")
    print(f"Baskets Discarded due to Illiquidity: {illiquid_count}")
    print(f"Baskets Discarded due to Charges > MinProfit: {charges_exceed_count}")
    print(f"Final Good Baskets Left: {final_good_baskets}")
    print("-" * 60)

    if not results:
        print("Search complete. No tradable risk-free baskets found after charges.")
        return

    # Create the dataframe and apply an explicit safety filter to ensure nothing sub-zero gets through to the plot
    results_df = pd.DataFrame(results)
    results_df = results_df[results_df['NetMinProfit'] > 0]
    
    if results_df.empty:
        print("Search complete. No tradable risk-free baskets found after charges.")
        return

    max_roi_row = results_df.loc[results_df['ROI'].idxmax()]

    def print_portfolio(row_data):
        for leg in row_data['Portfolio']:
            vol = 0
            if 'Volume' in df_analysis.columns:
                vol_mask = (df_analysis['Strike'] == leg['strike']) & \
                           (df_analysis['Type'] == leg['type']) & \
                           (pd.to_datetime(df_analysis['Expiry']).dt.date == pd.to_datetime(leg['expiry']).date())
                vol_data = df_analysis.loc[vol_mask, 'Volume']
                vol = int(vol_data.iloc[0]) if not vol_data.empty and not pd.isna(vol_data.iloc[0]) else 0
                
            print(f"      - {leg['action'].capitalize()} {leg['type']} @ {int(leg['strike'])} | Pts: ₹{leg['entry_price']:.2f} (Vol: {vol:,})")
        
        print(f"      => Net Premium: ₹{abs(row_data['TotalPremium']):,.2f} ({row_data['CreditDebit']})")

    print(f"🏆 BEST ROI BASKET:")
    print(f"   Expiry: {max_roi_row['Expiry']}")
    print_portfolio(max_roi_row)
    print(f"   ROI: {max_roi_row['ROI']:.2f}% | Margin: ₹{max_roi_row['Margin']:,.2f}")
    print(f"   Charges (Entry): ₹{max_roi_row['Charges']:,.2f}")
    print(f"   Net Guaranteed Profit: ₹{max_roi_row['NetMinProfit']:,.2f}")
    print("="*60 + "\n")

    print("Generating Plot...")
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=results_df['Expiry'],
        y=results_df['CenterStrike'],
        mode='markers',
        marker=dict(
            size=12,
            color=results_df['NetMinProfit'], 
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title="Net Guaranteed Profit (₹)")
        ),
        text=[
            f"<b>Expiry:</b> {r['Expiry']}<br>"
            f"<b>Avg Strike:</b> {r['CenterStrike']}<br>"
            f"<b>Legs:</b> {r['LegDetails']}<br>"
            f"<b>Charges:</b> ₹{r['Charges']:,.2f}<br>"
            f"<b>Net Min Profit:</b> ₹{r['NetMinProfit']:,.0f}<br>"
            f"<b>Net Max Profit:</b> ₹{r['NetMaxProfit']:,.0f}<br>"
            f"<b>Gross ROI:</b> {r['ROI']:.2f}%"
            for _, r in results_df.iterrows()
        ],
        hoverinfo='text',
        name='Risk-Free Baskets'
    ))

    fig.add_hline(y=spot_price, line_dash="dash", line_color="red", annotation_text=f"Spot: {spot_price}")

    fig.update_layout(
        title=f"Risk-Free Basket Opportunities (Buying Date: {buying_date} | Max {max_expiry_days} Days Out)",
        xaxis_title="Expiry Date",
        yaxis_title="Basket Avg Center Strike",
        height=700,
        template="plotly_white",
        xaxis=dict(type='category', categoryorder='category ascending')
    )

    fig.show()

# Execute the finder passing the max expiry days directly
find_arbitrage_baskets_plot(backtester, BUYING_DATE_INPUT, MAX_EXPIRY_DAYS_INPUT)

--- Initialization: Searching ALL 4-Leg Combinations on 2026-03-06 ---
Total theoretical basket combinations to test: 1,484,032


Checking Baskets:   0%|          | 0/1484032 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [6]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta
import sys
import os
import contextlib
from dotenv import load_dotenv
from tqdm.notebook import tqdm

# ==========================================
# 1. SETUP & IMPORTS
# ==========================================
# Add parent directory to path to import OptionsBacktester
try:
    from options_backtester import OptionsBacktester
except ImportError:
    print("❌ Error: Could not import OptionsBacktester. Make sure it exists in the parent directory.")

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ==========================================
# 2. DATA LOADING (Hardcoded)
# ==========================================
DATA_FILE_PATH = 'data/NIFTY50-INDEX_opt_2017-07-17_2026-02-27.csv'

# Load data via OptionsBacktester into global namespace to save time across runs
if 'backtester' not in globals():
    print(f"Initializing Backtester and loading data from {DATA_FILE_PATH}...")
    if os.path.exists(DATA_FILE_PATH):
        backtester = OptionsBacktester(data_path=DATA_FILE_PATH)
        df = backtester.df
        print("✅ Data loaded successfully.")
    else:
        raise FileNotFoundError(f"❌ Data file not found at {DATA_FILE_PATH}")
else:
    df = backtester.df

# ==========================================
# INPUTS
# ==========================================
START_DATE_INPUT = '2017-07-17'
INITIAL_CAPITAL = 100_000
MAX_EXPIRIES_AHEAD = 3      # Number of upcoming expiries to scan
MAX_SHIFT_PCT = 2.0         # Max % variation from Spot Price for the basket

# NEW PARAMETER: Date to unlock dynamic compounding. Trades before this use 1 lot.
DYNAMIC_SCALING_START_DATE = '2023-01-01' 

def calculate_trade_fees(leg_price, total_units, action, is_expiry=False, intrinsic_value=0):
    """
    Calculates exact NSE Options fees. 
    Crucially handles scaling: Brokerage is FLAT per leg, while STT/Txn scale with total units.
    """
    if not is_expiry:
        # --- ENTRY CHARGES ---
        turnover = leg_price * total_units
        brokerage = 20.0                                             # Flat 20 Rs per leg, regardless of qty
        txn_charge = turnover * 0.0003503                            # 0.03503% NSE transaction charge
        sebi = turnover * 0.000001                                   # ₹10 per crore
        ipft = turnover * 0.000005                                   # ₹50 per crore
        
        stamp_duty = turnover * 0.00003 if action == 'buy' else 0.0  # 0.003% on buy side only
        stt = turnover * 0.001 if action == 'sell' else 0.0          # 0.1% STT on options sell side
        
        gst = (brokerage + txn_charge + sebi) * 0.18                 # 18% GST
        
        return brokerage + txn_charge + sebi + ipft + stamp_duty + stt + gst
        
    else:
        # --- EXPIRY SETTLEMENT CHARGES ---
        stt = 0.0
        if action == 'buy' and intrinsic_value > 0:
            stt = (intrinsic_value * total_units) * 0.00125
            
        return stt

def run_dynamic_historical_backtest(start_date_str, max_expiries, max_shift_pct, dynamic_scaling_start_str):
    print(f"--- Starting Dynamic Historical Backtest from {start_date_str} (Including Indian Taxes/Fees) ---")

    load_dotenv()
    LOG_DIR = os.getenv("LOG_DIR", "logs")
    if not os.path.exists(LOG_DIR):
        os.makedirs(LOG_DIR)

    global df
    if 'Date' not in df.columns:
        df['Date'] = pd.to_datetime(df['Datetime']).dt.date
    
    df.sort_values(by=['Date', 'Expiry', 'Strike'], inplace=True)
    
    start_date = pd.to_datetime(start_date_str).normalize()
    dyn_start_ts = pd.to_datetime(dynamic_scaling_start_str).normalize()
    max_data_date = pd.to_datetime(df['Date'].max()).normalize()
    
    if start_date > max_data_date:
        print(f"❌ Start date {start_date.date()} is beyond available data (Max: {max_data_date.date()})")
        return

    # Simulation State Tracking (Two separate ledgers)
    current_date = start_date
    
    current_capital_flat = INITIAL_CAPITAL
    current_capital_dyn = INITIAL_CAPITAL
    
    trades_flat = []
    trades_dyn = []
    
    equity_curve = [{
        'date': start_date.date(), 
        'equity_flat': INITIAL_CAPITAL, 'return_pct_flat': 0.0,
        'equity_dyn': INITIAL_CAPITAL,  'return_pct_dyn': 0.0
    }]
    
    scan_stats = [] 
    pbar = tqdm(total=(max_data_date - start_date).days, desc="Backtesting Days")
    last_pbar_update = start_date

    # ==========================================
    # 3. HELPER FUNCTIONS
    # ==========================================
    def find_best_basket_for_date(analysis_date):
        """Scans for the highest Min Profit (Guaranteed) basket AFTER estimated 1-Lot fees."""
        df_day = df[df['Date'] == analysis_date].copy()
        if df_day.empty: 
            scan_stats.append({'Date': analysis_date.date(), 'BasketsFound': 0, 'Note': 'No Data'})
            return None

        spot_price = df_day['SpotPrice'].iloc[0] if 'SpotPrice' in df_day.columns else df_day['Close'].iloc[0]
        atm_strike = round(spot_price / 50) * 50
        
        legs_config = [
            {'offset': -50, 'type': 'PE', 'action': 'buy'},
            {'offset': 0,   'type': 'PE', 'action': 'sell'},
            {'offset': 50,  'type': 'CE', 'action': 'sell'},
            {'offset': 100, 'type': 'CE', 'action': 'buy'}
        ]

        valid_expiries = sorted([
            pd.to_datetime(d).normalize() for d in df_day['Expiry'].unique() 
            if analysis_date < pd.to_datetime(d).normalize()
        ])[:max_expiries] 
        
        if not valid_expiries: 
            scan_stats.append({'Date': analysis_date.date(), 'BasketsFound': 0, 'Note': 'No Expiries'})
            return None

        max_shift_abs = int(spot_price * (max_shift_pct / 100.0))
        max_shift_abs = round(max_shift_abs / 50) * 50  
        shift_range = range(-max_shift_abs, max_shift_abs + 50, 50)

        best_basket = None
        best_min_profit = -np.inf 
        baskets_found_count = 0

        with open(os.devnull, 'w') as devnull:
            for expiry in valid_expiries:
                expiry_str = str(expiry.date())
                
                for shift in shift_range:
                    portfolio = []
                    valid_legs = True
                    strikes_list = []
                    
                    for config in legs_config:
                        new_strike = atm_strike + config['offset'] + shift
                        strikes_list.append(new_strike)
                        
                        check_mask = (df_day['Expiry'] == expiry) & \
                                     (df_day['Strike'] == new_strike) & \
                                     (df_day['Type'] == config['type'])
                        
                        row = df_day[check_mask]
                        if row.empty:
                            valid_legs = False
                            break
                        
                        price = row['Close'].iloc[0]
                        lot_size = int(row['LotSize'].iloc[0]) if 'LotSize' in row.columns else 50
                        
                        portfolio.append({
                            'strike': new_strike,
                            'type': config['type'],
                            'action': config['action'],
                            'expiry': expiry_str,
                            'qty': 1,              
                            'entry_price': price,  
                            'lot_size': lot_size   
                        })
                    
                    if not valid_legs: continue

                    try:
                        with contextlib.redirect_stdout(devnull):
                            res = backtester.calculate_basket_payoff(
                                portfolio=portfolio, 
                                analysis_date=str(analysis_date.date()), 
                                show_plot=False, 
                                verbose=False
                            )
                            if res.get('status') != 'success':
                                continue
                            metrics = res['metrics']
                    except Exception:
                        continue
                    
                    proj_entry_fees = 0
                    proj_exit_fees_est = 0
                    
                    for leg in portfolio:
                        proj_entry_fees += calculate_trade_fees(leg['entry_price'], leg['lot_size'], leg['action'], False)
                        
                        if leg['type'] == 'CE': est_intrinsic = max(0, spot_price - leg['strike'])
                        else: est_intrinsic = max(0, leg['strike'] - spot_price)
                            
                        proj_exit_fees_est += calculate_trade_fees(0, leg['lot_size'], leg['action'], True, intrinsic_value=est_intrinsic)
                        
                    proj_total_fees = proj_entry_fees + proj_exit_fees_est
                    net_proj_min_profit = metrics['Max Loss'] - proj_total_fees
                    
                    if net_proj_min_profit > 0: 
                        baskets_found_count += 1
                        if net_proj_min_profit > best_min_profit:
                            best_min_profit = net_proj_min_profit
                            metrics['Max Loss'] = net_proj_min_profit 
                            best_basket = {
                                'portfolio': portfolio,
                                'metrics': metrics,
                                'expiry': expiry,
                                'strikes': strikes_list,
                                'atm_shift': shift,
                                'atm_strike': atm_strike,
                                'est_fees_1lot': proj_total_fees
                            }
        
        scan_stats.append({'Date': analysis_date.date(), 'BasketsFound': baskets_found_count, 'Note': 'Scanned'})
        return best_basket

    # ==========================================
    # 4. MAIN LOOP
    # ==========================================
    while current_date < max_data_date:
        days_diff = (current_date - last_pbar_update).days
        if days_diff > 0:
            pbar.update(days_diff)
            last_pbar_update = current_date
        
        best_trade = find_best_basket_for_date(current_date)
        
        if best_trade:
            portfolio_config = best_trade['portfolio']
            expiry_date = best_trade['expiry']
            proj_metrics = best_trade['metrics']
            safeguards = []
            
            # --- DYNAMIC POSITION SIZING ---
            margin_req_1_lot = proj_metrics['Margin Req']
            
            # Rule: Restrict to 1 lot prior to dynamic_scaling_start_str. 
            # Otherwise, scale if current capital exceeds the margin required for 2 lots.
            if current_date < dyn_start_ts:
                dyn_qty = 1
            else:
                if margin_req_1_lot > 0:
                    dyn_qty = max(1, int(current_capital_dyn // margin_req_1_lot))
                else:
                    dyn_qty = 1

            # --- SAME DAY ENTRY LOGIC ---
            open_position = []
            valid_entry = True
            
            flat_entry_fees = 0
            dyn_entry_fees = 0
            
            df_current_day = df[df['Date'] == current_date]
            
            for leg in portfolio_config:
                mask = (df_current_day['Strike'] == leg['strike']) & \
                       (df_current_day['Expiry'] == pd.to_datetime(leg['expiry'])) & \
                       (df_current_day['Type'] == leg['type'])
                row = df_current_day[mask]
                
                if row.empty:
                    valid_entry = False
                    break
                
                price = row['Close'].iloc[0]
                lot_size = int(row['LotSize'].iloc[0]) if 'LotSize' in row.columns else 50
                
                # Fees scale dynamically based on Total Units (qty * lot_size)
                flat_entry_fees += calculate_trade_fees(price, lot_size * 1, leg['action'], False)
                dyn_entry_fees += calculate_trade_fees(price, lot_size * dyn_qty, leg['action'], False)
                
                open_position.append({
                    'strike': leg['strike'],
                    'type': leg['type'],
                    'action': leg['action'],
                    'expiry': leg['expiry'],
                    'entry_price': price,
                    'lot_size': lot_size
                })

            if not valid_entry:
                current_date += timedelta(days=1)
                continue

            # --- FAST FORWARD TO EXPIRY ---
            exit_date = expiry_date
            is_unrealized = False
            
            if exit_date > max_data_date:
                exit_date = max_data_date
                is_unrealized = True

            pbar.update((exit_date - current_date).days)
            last_pbar_update = exit_date
            
            # --- EVALUATE PNL & FEES VIA SPOT PRICE ---
            flat_gross_pnl = 0
            dyn_gross_pnl = 0
            flat_exit_fees = 0
            dyn_exit_fees = 0
            
            spot_rows = df[df['Date'] == exit_date]
            
            if not spot_rows.empty and 'SpotPrice' in spot_rows.columns and not pd.isna(spot_rows['SpotPrice'].iloc[0]):
                spot_at_exit = spot_rows['SpotPrice'].iloc[0]
            elif not spot_rows.empty and 'Close' in spot_rows.columns:
                spot_at_exit = spot_rows['Close'].iloc[0]
            else:
                spot_at_exit = 0
                safeguards.append("Missing Expiry Spot")
            
            for leg in open_position:
                if leg['type'] == 'CE':
                    exit_price = max(0, spot_at_exit - leg['strike'])
                else:
                    exit_price = max(0, leg['strike'] - spot_at_exit)
                
                diff = exit_price - leg['entry_price']
                if leg['action'] == 'sell': diff = -diff
                
                # Gross PnL
                flat_gross_pnl += diff * (leg['lot_size'] * 1)
                dyn_gross_pnl += diff * (leg['lot_size'] * dyn_qty)
                
                # Exit Fees
                flat_exit_fees += calculate_trade_fees(0, leg['lot_size'] * 1, leg['action'], True, intrinsic_value=exit_price)
                dyn_exit_fees += calculate_trade_fees(0, leg['lot_size'] * dyn_qty, leg['action'], True, intrinsic_value=exit_price)

            # --- COMPILE FLAT TRADES ---
            flat_total_fees = flat_entry_fees + flat_exit_fees
            flat_net_pnl = flat_gross_pnl - flat_total_fees
            
            trades_flat.append({
                'Date': current_date.date(),
                'Exit': exit_date.date(),
                'Lots': 1,
                'Type': 'Realized' if not is_unrealized else 'Unrealized',
                'Strikes': str(best_trade['strikes']),
                'ATM Shift': best_trade['atm_shift'],  
                'Days': (exit_date - current_date).days,
                'Margin': margin_req_1_lot,
                'Fees': flat_total_fees,
                'Gross PnL': flat_gross_pnl,
                'Net PnL': flat_net_pnl,
                'ROI': (flat_net_pnl / margin_req_1_lot * 100) if margin_req_1_lot > 0 else 0,
                'Issues': " | ".join(set(safeguards)) if safeguards else "-"
            })
            
            # --- COMPILE DYNAMIC TRADES ---
            dyn_total_fees = dyn_entry_fees + dyn_exit_fees
            dyn_net_pnl = dyn_gross_pnl - dyn_total_fees
            dyn_margin_req = margin_req_1_lot * dyn_qty
            
            trades_dyn.append({
                'Date': current_date.date(),
                'Exit': exit_date.date(),
                'Lots': dyn_qty,
                'Type': 'Realized' if not is_unrealized else 'Unrealized',
                'Strikes': str(best_trade['strikes']),
                'ATM Shift': best_trade['atm_shift'],  
                'Days': (exit_date - current_date).days,
                'Margin': dyn_margin_req,
                'Fees': dyn_total_fees,
                'Gross PnL': dyn_gross_pnl,
                'Net PnL': dyn_net_pnl,
                'ROI': (dyn_net_pnl / dyn_margin_req * 100) if dyn_margin_req > 0 else 0,
                'Issues': " | ".join(set(safeguards)) if safeguards else "-"
            })
            
            current_capital_flat += flat_net_pnl
            current_capital_dyn += dyn_net_pnl
            
            equity_curve.append({
                'date': exit_date.date(),
                'equity_flat': current_capital_flat,
                'return_pct_flat': ((current_capital_flat - INITIAL_CAPITAL) / INITIAL_CAPITAL) * 100,
                'equity_dyn': current_capital_dyn,
                'return_pct_dyn': ((current_capital_dyn - INITIAL_CAPITAL) / INITIAL_CAPITAL) * 100
            })
            
            current_date = exit_date + timedelta(days=1)
            
        else:
            current_date += timedelta(days=1)

    pbar.close()

    # ==========================================
    # 5. REPORTING & LOGGING
    # ==========================================
    if not trades_flat:
        print("No trades found matching criteria (All opportunities destroyed by fees).")
        return

    df_flat = pd.DataFrame(trades_flat)
    df_dyn = pd.DataFrame(trades_dyn)
    
    total_days = (max_data_date - start_date).days
    years = total_days / 365.25
    
    def generate_summary(df_trades, capital_final, strategy_name):
        total_count = len(df_trades)
        win_count = len(df_trades[df_trades['Net PnL'] > 0])
        win_rate = win_count / total_count * 100 if total_count > 0 else 0
        
        avg_win = df_trades[df_trades['Net PnL'] > 0]['Net PnL'].mean()
        avg_loss = df_trades[df_trades['Net PnL'] <= 0]['Net PnL'].mean()
        avg_hold = df_trades['Days'].mean()
        
        net_pnl = capital_final - INITIAL_CAPITAL
        total_return = (net_pnl / INITIAL_CAPITAL) * 100
        annualized_return = ((1 + total_return/100) ** (1/years) - 1) * 100 if years > 0 else 0
        
        # Max Drawdown Calculation
        equity_series = pd.DataFrame(equity_curve)[f'equity_{"dyn" if "Dynamic" in strategy_name else "flat"}']
        rolling_max = equity_series.cummax()
        drawdown = (equity_series - rolling_max) / rolling_max
        max_dd = drawdown.min() * 100

        text = (
            f"--- {strategy_name.upper()} ---\n"
            f"Final Capital:    {capital_final:,.2f}  (Start: {INITIAL_CAPITAL:,.2f})\n"
            f"Net Profit:       {net_pnl:,.2f}\n"
            f"Total Return:     {total_return:.2f}%\n"
            f"Annualized CAGR:  {annualized_return:.2f}%\n"
            f"Max Drawdown:     {max_dd:.2f}%\n"
            f"Win Rate:         {win_rate:.2f}% ({win_count}/{total_count} Trades)\n"
            f"Avg Win/Loss:     {avg_win:,.2f} / {avg_loss:,.2f}\n"
            f"Avg Hold Time:    {avg_hold:.1f} days\n"
        )
        return text

    sum_flat = generate_summary(df_flat, current_capital_flat, "Flat 1-Lot Strategy")
    sum_dyn = generate_summary(df_dyn, current_capital_dyn, "Dynamic Compounding Strategy")

    master_summary = (
        f"BACKTEST SUMMARY REPORT (FEES INCLUDED)\n"
        f"=======================================\n"
        f"Strategy:         Risk-Free Arbitrage (Same Day Entry & Intrinsic Exit)\n"
        f"Start Date:       {start_date.date()}\n"
        f"End Date:         {max_data_date.date()}\n"
        f"Duration:         {years:.2f} years\n"
        f"Scaling Unlock:   {pd.to_datetime(dynamic_scaling_start_str).date()}\n"
        f"=======================================\n\n"
        f"{sum_flat}\n"
        f"---------------------------------------\n\n"
        f"{sum_dyn}\n"
        f"=======================================\n"
    )

    log_filename_flat = os.path.join(RESULTS_DIR, f"trade_log_{start_date_str.replace('-','')}_flat.txt")
    log_filename_dyn = os.path.join(RESULTS_DIR, f"trade_log_{start_date_str.replace('-','')}_dyn.txt")
    
    cols_for_log = ['Date', 'Exit', 'Lots', 'ATM Shift', 'Strikes', 'Days', 'Margin', 'Fees', 'Gross PnL', 'Net PnL', 'ROI', 'Issues']
    
    with open(log_filename_flat, 'w') as f:
        f.write(master_summary)
        f.write("\nFLAT 1-LOT TRADE LOG:\n" + "-" * 30 + "\n")
        f.write(df_flat[cols_for_log].to_markdown(index=False, numalign="right", floatfmt=".2f"))
        
    with open(log_filename_dyn, 'w') as f:
        f.write(master_summary)
        f.write("\nDYNAMIC COMPOUNDING TRADE LOG:\n" + "-" * 30 + "\n")
        f.write(df_dyn[cols_for_log].to_markdown(index=False, numalign="right", floatfmt=".2f"))
        
    print(f"\n✅ Logs saved to:\n  - {log_filename_flat}\n  - {log_filename_dyn}")
    print("\n" + master_summary)

    # ==========================================
    # 6. PLOTTING (Two Separate Graphs)
    # ==========================================
    df_eq = pd.DataFrame(equity_curve)

    # Plot 1: Flat 1-Lot Equity
    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(
        x=df_eq['date'], y=df_eq['return_pct_flat'],
        mode='lines+markers', name='Flat 1-Lot Equity',
        line=dict(color='blue', width=2), fill='tozeroy', fillcolor='rgba(0, 0, 255, 0.1)'
    ))
    fig1.update_layout(
        title=f"Net Equity Curve - FLAT 1-LOT (Start: {start_date.date()})",
        xaxis_title="Date", yaxis_title="Return (%)",
        template="plotly_white", height=500, showlegend=False
    )
    fig1.add_hline(y=0, line_dash="dash", line_color="black")
    fig1.show()

    # Plot 2: Dynamic Compounding Equity
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(
        x=df_eq['date'], y=df_eq['return_pct_dyn'],
        mode='lines+markers', name='Dynamic Compounding Equity',
        line=dict(color='green', width=2), fill='tozeroy', fillcolor='rgba(0, 128, 0, 0.1)'
    ))
    fig2.update_layout(
        title=f"Net Equity Curve - DYNAMIC COMPOUNDING (Unlocked: {pd.to_datetime(dynamic_scaling_start_str).date()})",
        xaxis_title="Date", yaxis_title="Return (%)",
        template="plotly_white", height=500, showlegend=False
    )
    fig2.add_hline(y=0, line_dash="dash", line_color="black")
    fig2.show()

# Run
run_dynamic_historical_backtest(START_DATE_INPUT, MAX_EXPIRIES_AHEAD, MAX_SHIFT_PCT, DYNAMIC_SCALING_START_DATE)

--- Starting Dynamic Historical Backtest from 2017-07-17 (Including Indian Taxes/Fees) ---


Backtesting Days:   0%|          | 0/3203 [00:00<?, ?it/s]


✅ Logs saved to:
  - results/trade_log_20170717_flat.txt
  - results/trade_log_20170717_dyn.txt

BACKTEST SUMMARY REPORT (FEES INCLUDED)
Strategy:         Risk-Free Arbitrage (Same Day Entry & Intrinsic Exit)
Start Date:       2017-07-17
End Date:         2026-04-24
Duration:         8.77 years
Scaling Unlock:   2023-01-01

--- FLAT 1-LOT STRATEGY ---
Final Capital:    303,478.04  (Start: 100,000.00)
Net Profit:       203,478.04
Total Return:     203.48%
Annualized CAGR:  13.50%
Max Drawdown:     -0.51%
Win Rate:         98.10% (155/158 Trades)
Avg Win/Loss:     1,319.68 / -357.61
Avg Hold Time:    17.2 days

---------------------------------------

--- DYNAMIC COMPOUNDING STRATEGY ---
Final Capital:    1,431,234.70  (Start: 100,000.00)
Net Profit:       1,331,234.70
Total Return:     1331.23%
Annualized CAGR:  35.45%
Max Drawdown:     -2.30%
Win Rate:         98.10% (155/158 Trades)
Avg Win/Loss:     8,625.47 / -1,904.47
Avg Hold Time:    17.2 days




In [8]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import os
import contextlib
from tqdm import tqdm

# ==========================================
# 1. SETUP & INPUTS
# ==========================================
START_DATE_INPUT = '2017-07-17'
INITIAL_CAPITAL = 500_000
IRON_CONDOR_RANGE = 200     
DYNAMIC_SCALING_START_DATE = '2020-01-01' 

DATA_FILE_PATH = 'data/NIFTY50-INDEX_opt_2017-07-17_2026-04-24.csv'
UNDERLYING_5M_PATH = 'data/nifty50_5m_20170717_20260424.csv'

def calculate_trade_fees(price, total_units, action):
    """Calculates NSE Options fees including STT, Brokerage, and GST."""
    turnover = price * total_units
    brokerage = 20.0
    txn_charge = turnover * 0.00053 
    stt = turnover * 0.0005 if action == 'sell' else 0.0
    gst = (brokerage + txn_charge) * 0.18
    return brokerage + txn_charge + stt + gst

def generate_full_summary(df_trades, initial_cap, final_cap, strategy_name, start_date, end_date):
    if df_trades.empty: 
        return f"--- {strategy_name.upper()} ---\nNo trades were generated. Ensure Date coverage matches."
    
    total_net_pnl = final_cap - initial_cap
    total_return = (total_net_pnl / initial_cap) * 100
    
    # CAGR Calculation
    years = (end_date - start_date).days / 365.25
    cagr = ((final_cap / initial_cap) ** (1/years) - 1) * 100 if years > 0 else 0
    
    # Max Drawdown Calculation
    equity_series = df_trades['Cumulative_Equity']
    peak = equity_series.cummax()
    drawdown = (equity_series - peak) / peak
    max_dd = drawdown.min() * 100
    
    # Trade Stats
    wins = df_trades[df_trades['Net PnL'] > 0]
    losses = df_trades[df_trades['Net PnL'] <= 0]
    win_rate = (len(wins) / len(df_trades)) * 100
    avg_win = wins['Net PnL'].mean() if not wins.empty else 0
    avg_loss = losses['Net PnL'].mean() if not losses.empty else 0
    
    return (f"--- {strategy_name.upper()} ---\n"
            f"Final Capital:    {final_cap:,.2f}  (Start: {initial_cap:,.2f})\n"
            f"Net Profit:       {total_net_pnl:,.2f}\n"
            f"Total Return:     {total_return:.2f}%\n"
            f"Annualized CAGR:  {cagr:.2f}%\n"
            f"Max Drawdown:     {max_dd:.2f}%\n"
            f"Win Rate:         {win_rate:.2f}% ({len(wins)}/{len(df_trades)} Trades)\n"
            f"Avg Win/Loss:     {avg_win:,.2f} / {avg_loss:,.2f}\n"
            f"Avg Hold Time:    Intraday (1 Day)\n")

# ==========================================
# 2. OPTIMIZED ENGINE
# ==========================================
def run_iron_condor_pro_backtest():
    print("Loading and indexing options data... (O(1) lookup optimization)")
    
    # Load Options Data
    df_opt = pd.read_csv(DATA_FILE_PATH)
    df_opt['DateOnly'] = pd.to_datetime(df_opt['Datetime']).dt.date
    
    # Create high-speed lookup dictionary
    # Maps (Date, Strike, Type, Expiry) -> (Open, Close, LotSize)
    # We map 'C' -> 'CE' and 'P' -> 'PE' if necessary to match logic
    opt_lookup = {}
    for row in df_opt.itertuples():
        # Standardizing Type to CE/PE for the strategy logic
        t = 'CE' if row.Type in ['C', 'CE'] else 'PE'
        opt_lookup[(row.DateOnly, float(row.Strike), t, str(row.Expiry))] = (row.Open, row.Close, row.LotSize)
    
    # Load and process Underlying Data
    df_und = pd.read_csv(UNDERLYING_5M_PATH)
    df_und['Datetime'] = pd.to_datetime(df_und['Datetime'])
    df_und['DateOnly'] = df_und['Datetime'].dt.date
    
    # Vectorized: Get only the first candle (Market Open) for each day
    df_openings = df_und.sort_values('Datetime').groupby('DateOnly').first().reset_index()
    unique_days = df_openings[df_openings['DateOnly'] >= pd.to_datetime(START_DATE_INPUT).date()]
    
    dyn_start_ts = pd.to_datetime(DYNAMIC_SCALING_START_DATE).date()

    cap_flat, cap_dyn = INITIAL_CAPITAL, INITIAL_CAPITAL
    trades_flat, trades_dyn = [], []

    print(f"Starting simulation for {len(unique_days)} days...")
    
    for day_row in tqdm(unique_days.itertuples(), total=len(unique_days)):
        current_date = day_row.DateOnly
        atm_strike = round(float(day_row.Open) / 50) * 50
        
        # Get expiry available on this specific date from our lookup keys
        possible_expiries = sorted({k[3] for k in opt_lookup.keys() if k[0] == current_date})
        if not possible_expiries: continue
        expiry = possible_expiries[0] # Front-month expiry

        # Short Iron Condor Structure
        leg_configs = [
            (atm_strike - (IRON_CONDOR_RANGE/2) - 50, 'PE', 'buy'),
            (atm_strike - (IRON_CONDOR_RANGE/2),      'PE', 'sell'),
            (atm_strike + (IRON_CONDOR_RANGE/2),      'CE', 'sell'),
            (atm_strike + (IRON_CONDOR_RANGE/2) + 50, 'CE', 'buy')
        ]

        day_legs = []
        valid_basket = True
        for strike, op_type, action in leg_configs:
            key = (current_date, float(strike), op_type, expiry)
            if key in opt_lookup:
                day_legs.append({
                    'entry': opt_lookup[key][0], 
                    'exit': opt_lookup[key][1], 
                    'lot': opt_lookup[key][2], 
                    'action': action
                })
            else:
                valid_basket = False
                break

        if not valid_basket: continue

        # Position Sizing based on Margin (approx 1.5L for Hedged Nifty Basket)
        margin_1lot = 150000 
        dyn_qty = max(1, int(cap_dyn // margin_1lot)) if current_date >= dyn_start_ts else 1

        net_1lot_day, net_dyn_day = 0, 0
        
        for leg in day_legs:
            pnl_unit = (leg['exit'] - leg['entry']) if leg['action'] == 'buy' else (leg['entry'] - leg['exit'])
            
            # Flat 1-Lot Strategy
            f_entry_1 = calculate_trade_fees(leg['entry'], leg['lot'], leg['action'])
            f_exit_1 = calculate_trade_fees(leg['exit'], leg['lot'], 'sell' if leg['action'] == 'buy' else 'buy')
            net_1lot_day += (pnl_unit * leg['lot']) - (f_entry_1 + f_exit_1)
            
            # Dynamic Compounding Strategy
            f_entry_d = calculate_trade_fees(leg['entry'], leg['lot'] * dyn_qty, leg['action'])
            f_exit_d = calculate_trade_fees(leg['exit'], leg['lot'] * dyn_qty, 'sell' if leg['action'] == 'buy' else 'buy')
            net_dyn_day += (pnl_unit * leg['lot'] * dyn_qty) - (f_entry_d + f_exit_d)

        cap_flat += net_1lot_day
        cap_dyn += net_dyn_day

        trades_flat.append({'Date': current_date, 'Net PnL': net_1lot_day, 'Cumulative_Equity': cap_flat})
        trades_dyn.append({'Date': current_date, 'Net PnL': net_dyn_day, 'Cumulative_Equity': cap_dyn})

    # ==========================================
    # 3. REPORTING & VISUALIZATION
    # ==========================================
    df_f = pd.DataFrame(trades_flat)
    df_d = pd.DataFrame(trades_dyn)
    
    if df_f.empty:
        print("❌ ERROR: No trades found. Check if Options and Underlying data dates overlap.")
        return

    # Print Summaries
    print("\n" + generate_full_summary(df_f, INITIAL_CAPITAL, cap_flat, "Flat 1-Lot Strategy", df_f['Date'].min(), df_f['Date'].max()))
    print("\n" + generate_full_summary(df_d, INITIAL_CAPITAL, cap_dyn, "Dynamic Compounding Strategy", df_d['Date'].min(), df_d['Date'].max()))

    # Plot Equity Curves
    for df_plt, name in [(df_f, "Flat"), (df_d, "Dynamic")]:
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=df_plt['Date'], y=df_plt['Cumulative_Equity'], name=name, line=dict(width=2)))
        fig.update_layout(
            title=f"Equity Curve: {name} Strategy",
            xaxis_title="Date", 
            yaxis_title="Total Capital (₹)",
            template="plotly_white",
            hovermode="x unified"
        )
        fig.show()

# Execution
if __name__ == "__main__":
    run_iron_condor_pro_backtest()

Loading and indexing options data... (O(1) lookup optimization)


KeyboardInterrupt: 

## Run to restore backed up csv file

In [ ]:
import os
import shutil
from dotenv import load_dotenv

# 1. Load Configuration
load_dotenv()
DATA_PATH = os.getenv("OPTIONS_DATA_FILE", "data/NIFTY_opt_final.csv")

# 2. Define Paths
original_csv = DATA_PATH
backup_file = DATA_PATH + ".bak"

def restore_from_backup():
    print(f"--- Restoring Data from Backup ---")
    print(f"Target: {original_csv}")
    print(f"Source: {backup_file}")
    
    # Check if backup exists
    if not os.path.exists(backup_file):
        print(f"❌ Error: Backup file not found at {backup_file}")
        return

    try:
        # Check if the current file exists (to print its size for comparison)
        if os.path.exists(original_csv):
            current_size = os.path.getsize(original_csv) / (1024 * 1024)
            print(f"Current File Size: {current_size:.2f} MB")
        
        backup_size = os.path.getsize(backup_file) / (1024 * 1024)
        print(f"Backup File Size:  {backup_size:.2f} MB")
        
        # RESTORE: Copy .bak -> .csv
        shutil.copy(backup_file, original_csv)
        
        print(f"\n✅ Success! Restored {original_csv} from backup.")
        print("You can now re-run the process with the corrected code.")
        
    except Exception as e:
        print(f"❌ Failed to restore: {e}")


restore_from_backup()

--- Restoring Data from Backup ---
Target: data/NIFTY_opt_2019-01-01_2026-02-12.csv
Source: data/NIFTY_opt_2019-01-01_2026-02-12.csv.bak
Current File Size: 1037.28 MB
Backup File Size:  643.79 MB

✅ Success! Restored data/NIFTY_opt_2019-01-01_2026-02-12.csv from backup.
You can now re-run the process with the corrected code.


In [ ]:
import pandas as pd
import numpy as np
import os
import sys
from datetime import date
from tqdm import tqdm
from dotenv import load_dotenv

# ==========================================
# 1. CONFIGURATION & SETUP
# ==========================================
load_dotenv()
FILE_PATH = os.getenv("OPTIONS_DATA_FILE", "nifty_options_data.csv")
TEMP_FILE = FILE_PATH + ".tmp"

# ==========================================
# 2. DEFINE LOT SIZE LOGIC
# ==========================================
def get_historical_lot_size(symbol, trade_date):
    """
    Returns the lot size based on historical exchange rules.
    """
    # SAFETY: Check for NaT (Invalid Date) or None
    if pd.isnull(trade_date):
        return 0
        
    # Handle both string dates and Timestamp objects
    try:
        if isinstance(trade_date, pd.Timestamp):
            dt = trade_date.date()
        else:
            dt = pd.to_datetime(trade_date).date()
    except:
        return 0 # Fail safe
    
    # Normalize Symbol (Handle "NSE:NIFTY50-INDEX" -> "NIFTY")
    if "NIFTY" in str(symbol).upper() and "BANK" not in str(symbol).upper():
        clean_symbol = "NIFTY"
    elif "BANKNIFTY" in str(symbol).upper() or "NIFTYBANK" in str(symbol).upper():
        clean_symbol = "BANKNIFTY"
    else:
        clean_symbol = symbol

    # --- HISTORICAL RULES ---
    if clean_symbol == "NIFTY":
        if dt < date(2024, 5, 1): return 50
        elif dt < date(2024, 12, 1): return 25
        elif dt < date(2026, 1, 1): return 75
        else: return 65 
        
    elif clean_symbol == "BANKNIFTY":
        if dt < date(2020, 1, 1): return 20 
        elif dt < date(2023, 7, 1): return 25
        elif dt < date(2025, 1, 1): return 15
        elif dt < date(2025, 7, 31): return 30
        elif dt < date(2026, 1, 1): return 35
        else: return 30 
        
    return 0 

# ==========================================
# 3. EXECUTION (ATOMIC WRITE)
# ==========================================
if not os.path.exists(FILE_PATH):
    print(f"❌ Error: File not found at {FILE_PATH}")
else:
    try:
        print(f"Loading data from {FILE_PATH}...")
        df = pd.read_csv(FILE_PATH)
        
        # Ensure Datetime column is in correct format for the function
        df['Datetime'] = pd.to_datetime(df['Datetime'])
        
        print("Calculating Historical Lot Sizes...")
        tqdm.pandas(desc="Processing Rows")
        
        # Apply the logic row-by-row
        df['LotSize'] = df.progress_apply(
            lambda row: get_historical_lot_size(row['Symbol'], row['Datetime']), 
            axis=1
        )
        
        # Sanity Check
        zero_lots = (df['LotSize'] == 0).sum()
        if zero_lots > 0:
            print(f"⚠️ Warning: {zero_lots} rows have 0 LotSize (Symbol mismatch or bad date).")
        
        print(f"Saving to temp file: {TEMP_FILE}...")
        df.to_csv(TEMP_FILE, index=False)
        
        # Atomic Swap
        if os.path.exists(TEMP_FILE):
            if os.path.exists(FILE_PATH):
                os.remove(FILE_PATH)
            os.rename(TEMP_FILE, FILE_PATH)
            print(f"✅ SUCCESS. File updated safely: {FILE_PATH}")
            print(f"Sample Lot Sizes:\n{df[['Datetime', 'Symbol', 'LotSize']].head()}")
        else:
            print("❌ Error: Temp file creation failed.")
            
    except Exception as e:
        print(f"\n❌ CRITICAL ERROR: {e}")
        # Cleanup temp file if it exists to avoid clutter
        if os.path.exists(TEMP_FILE):
            os.remove(TEMP_FILE)

Loading data from data/NIFTY_opt_2017-07-17_2026-02-12.csv...
Calculating Historical Lot Sizes...


Processing Rows: 100%|██████████| 4562704/4562704 [00:24<00:00, 187292.67it/s]


Saving to temp file: data/NIFTY_opt_2017-07-17_2026-02-12.csv.tmp...
✅ SUCCESS. File updated safely: data/NIFTY_opt_2017-07-17_2026-02-12.csv
Sample Lot Sizes:
    Datetime Symbol  LotSize
0 2017-07-17  NIFTY       50
1 2017-07-18  NIFTY       50
2 2017-07-19  NIFTY       50
3 2017-07-20  NIFTY       50
4 2017-07-21  NIFTY       50
